In [33]:
import pandas as pd

# === Load raw data ===
df = pd.read_csv('../merged_output/VAERS_MODEL_READY.csv', low_memory=False)
# 1. Outcome columns → Yes/No only
# ======================
def outcome_binary(val):
    v = str(val).strip().upper() if pd.notna(val) else ""
    if v in {"Y", "YES", "1", "TRUE", "T"}:
        return "Yes"
    else:
        return "No"   # treat everything else as No

for col in ["DIED", "L_THREAT", "DISABLE", "RECOVD"]:
    if col in df.columns:
        df[col] = df[col].apply(outcome_binary)

# ======================
# 2. Normalize SEX
# ======================
if "SEX" in df.columns:
    df["SEX"] = df["SEX"].fillna("Unknown").str.upper()
    df["SEX"] = df["SEX"].replace({"U": "Unknown", "": "Unknown"})

# ======================
# 3. Normalize VAX_DOSE_SERIES
# ======================
if "VAX_DOSE_SERIES" in df.columns:
    df["VAX_DOSE_SERIES"] = (
        df["VAX_DOSE_SERIES"]
        .fillna("UNK")
        .astype(str)
        .str.strip()
        .replace({"": "UNK"})
    )

# ======================
# 4. Build and clean symptoms
# ======================
if {"SYMPTOM1", "SYMPTOM2", "SYMPTOM3"}.issubset(df.columns):
    # If dataset has SYMPTOM1..3
    df["symptoms_list"] = df[["SYMPTOM1", "SYMPTOM2", "SYMPTOM3"]].values.tolist()
    df["symptoms_list"] = df["symptoms_list"].apply(
        lambda lst: [s for s in lst if pd.notna(s)]
    )
elif "symptoms_list" in df.columns:
    # Already exists, make sure it's a list
    df["symptoms_list"] = df["symptoms_list"].apply(
        lambda x: eval(x) if isinstance(x, str) and x.startswith("[") else x
    )
else:
    df["symptoms_list"] = [[] for _ in range(len(df))]

# Deduplicate + lowercase
df["symptoms_list"] = df["symptoms_list"].apply(
    lambda lst: sorted(set([s.strip().lower() for s in lst if isinstance(s, str)]))
    if isinstance(lst, list) else []
)

# Normalized string version
df["symptoms_normalized"] = df["symptoms_list"].apply(lambda lst: "; ".join(lst))

# ======================
# 5. Age bucket
# ======================
def age_bucket(age):
    try:
        a = float(age)
    except:
        return "Unknown"
    if a < 5: return "0-4"
    if a < 18: return "5-17"
    if a < 50: return "18-49"
    if a < 65: return "50-64"
    return "65+"

if "AGE_YRS" in df.columns:
    df["age_bucket"] = df["AGE_YRS"].apply(age_bucket)
else:
    df["age_bucket"] = "Unknown"

# ======================
# 6. Any serious flag
# ======================
df["any_serious"] = df[["DIED","L_THREAT","DISABLE"]].apply(
    lambda row: "Yes" if "Yes" in row.values else "No",
    axis=1
)

# ======================
# 7. Keep only useful columns
# ======================
keep_cols = [
    "VAERS_ID", "AGE_YRS", "SEX", "STATE", "YEAR",
    "VAX_TYPE", "VAX_MANU", "VAX_DOSE_SERIES",
    "DIED", "L_THREAT", "DISABLE", "RECOVD",
    "symptoms_list", "age_bucket", "any_serious", "symptoms_normalized"
]
df_clean = df[[c for c in keep_cols if c in df.columns]].copy()

# ======================
# 8. Save
# ======================
df_clean.to_csv("vaers_sample_cleaned.csv", index=False)

print("✅ Cleaned dataset saved as vaers_sample_cleaned.csv")
print(df_clean.head())

✅ Cleaned dataset saved as vaers_sample_cleaned.csv
   VAERS_ID  AGE_YRS SEX STATE  YEAR VAX_TYPE                     VAX_MANU  \
0    810053     73.0   F    OH  2020     FLU3               SANOFI PASTEUR   
1    810053     73.0   F    OH  2020     FLU3               SANOFI PASTEUR   
2    855017     55.0   F    HI  2020   VARZOS  GLAXOSMITHKLINE BIOLOGICALS   
3    855018     68.0   F    WI  2020      UNK         UNKNOWN MANUFACTURER   
4    855018     68.0   F    WI  2020      UNK         UNKNOWN MANUFACTURER   

  VAX_DOSE_SERIES DIED L_THREAT DISABLE RECOVD  \
0             UNK   No       No      No     No   
1             UNK   No       No      No     No   
2               2   No       No      No     No   
3               1   No       No      No    Yes   
4               1   No       No      No    Yes   

                                     symptoms_list age_bucket any_serious  \
0                     [asthenia, chills, headache]        65+          No   
1                       